In [ ]:
import cv2
import numpy as np
import os
from glob import glob

CHECKERBOARD = (8, 6)
LEFT_IMAGES_DIR = './left_frames'
RIGHT_IMAGES_DIR = './right_frames'
OUTPUT_DIR = 'output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Step 1: Stereo Calibration
def stereo_calibrate():
    # Prepare object points
    objp = np.zeros((CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
    objp[:, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)

    objpoints = []  # 3D points
    imgpoints_left = []  # 2D points left
    imgpoints_right = []  # 2D points right

    images_left = sorted(glob(os.path.join(LEFT_IMAGES_DIR, '*.png')))
    images_right = sorted(glob(os.path.join(RIGHT_IMAGES_DIR, '*.png')))

    if not images_left or not images_right:
        raise ValueError("No images found in the left or right directory.")

    for left_img_path, right_img_path in zip(images_left, images_right):
        img_left = cv2.imread(left_img_path)
        img_right = cv2.imread(right_img_path)

        gray_left = cv2.cvtColor(img_left, cv2.COLOR_BGR2GRAY)
        gray_right = cv2.cvtColor(img_right, cv2.COLOR_BGR2GRAY)

        ret_left, corners_left = cv2.findChessboardCorners(gray_left, CHECKERBOARD, None)
        ret_right, corners_right = cv2.findChessboardCorners(gray_right, CHECKERBOARD, None)

        if ret_left and ret_right:
            objpoints.append(objp)
            imgpoints_left.append(corners_left)
            imgpoints_right.append(corners_right)

    # Check if we found enough points
    if len(objpoints) == 0:
        raise ValueError("No valid checkerboard detections. Please check your calibration images.")

    # Proceed with calibration
    image_size = gray_left.shape[::-1]

    ret_left, mtx_left, dist_left, _, _ = cv2.calibrateCamera(objpoints, imgpoints_left, image_size, None, None)
    ret_right, mtx_right, dist_right, _, _ = cv2.calibrateCamera(objpoints, imgpoints_right, image_size, None, None)

    # Stereo calibration
    flags = cv2.CALIB_FIX_INTRINSIC
    _, _, _, _, _, R, T, E, F = cv2.stereoCalibrate(
        objpoints, imgpoints_left, imgpoints_right,
        mtx_left, dist_left,
        mtx_right, dist_right,
        image_size, criteria=(cv2.TERM_CRITERIA_MAX_ITER + cv2.TERM_CRITERIA_EPS, 100, 1e-5), flags=flags
    )

    # Stereo rectification
    R1, R2, P1, P2, Q, _, _ = cv2.stereoRectify(
        mtx_left, dist_left, mtx_right, dist_right,
        image_size, R, T, alpha=0
    )

    return mtx_left, dist_left, mtx_right, dist_right, R1, R2, P1, P2, Q

# Step 2: Compute depth map
def compute_depth_map(img_left, img_right):
    gray_left = cv2.cvtColor(img_left, cv2.COLOR_BGR2GRAY)
    gray_right = cv2.cvtColor(img_right, cv2.COLOR_BGR2GRAY)

    # Stereo matcher
    stereo = cv2.StereoBM_create(numDisparities=16 * 5, blockSize=15)
    disparity = stereo.compute(gray_left, gray_right).astype(np.float32) / 16.0

    # Optional filtering (median)
    disparity = cv2.medianBlur(disparity, 5)

    return disparity

# Step 3: Export point cloud
def export_point_cloud(img_left, disparity, Q):
    points_3D = cv2.reprojectImageTo3D(disparity, Q)
    colors = cv2.cvtColor(img_left, cv2.COLOR_BGR2RGB)

    mask = disparity > disparity.min()
    output_points = points_3D[mask]
    output_colors = colors[mask]

    ply_header = '''ply
format ascii 1.0
element vertex {vertex_count}
property float x
property float y
property float z
property uchar red
property uchar green
property uchar blue
end_header
'''

    with open(os.path.join(OUTPUT_DIR, 'point_cloud.ply'), 'w') as f:
        f.write(ply_header.format(vertex_count=len(output_points)))
        for p, c in zip(output_points, output_colors):
            f.write('%f %f %f %d %d %d\n' % (*p, *c))

# Step 4: Detect distance and warn
def detect_distance(disparity, Q):
    points_3D = cv2.reprojectImageTo3D(disparity, Q)
    mask = disparity > disparity.min()

    distances = np.linalg.norm(points_3D[mask], axis=1)
    if np.any(distances < 500):  # mm, so 500 mm = 50 cm
        print("⚠️ Warning: Object too close!")

# Main pipeline
def main():
    mtx_left, dist_left, mtx_right, dist_right, R1, R2, P1, P2, Q = stereo_calibrate()

    # Pick one pair for depth computation
    img_left = cv2.imread(sorted(glob(os.path.join(LEFT_IMAGES_DIR, '*.png')))[0])
    img_right = cv2.imread(sorted(glob(os.path.join(RIGHT_IMAGES_DIR, '*.png')))[0])

    # Rectify images
    h, w = img_left.shape[:2]
    map1_left, map2_left = cv2.initUndistortRectifyMap(mtx_left, dist_left, R1, P1, (w, h), cv2.CV_16SC2)
    map1_right, map2_right = cv2.initUndistortRectifyMap(mtx_right, dist_right, R2, P2, (w, h), cv2.CV_16SC2)

    rectified_left = cv2.remap(img_left, map1_left, map2_left, cv2.INTER_LINEAR)
    rectified_right = cv2.remap(img_right, map1_right, map2_right, cv2.INTER_LINEAR)

    # Compute depth
    disparity = compute_depth_map(rectified_left, rectified_right)

    # Export point cloud
    export_point_cloud(rectified_left, disparity, Q)

    # Detect distance
    detect_distance(disparity, Q)

    # Optional: visualize depth map
    cv2.imshow('Disparity', (disparity - disparity.min()) / (disparity.max() - disparity.min()))
    cv2.waitKey(0)
    cv2.destroyAllWindows()

if __name__ == '__main__':
    main()


/var/folders/d5/z9jzrbjx4bncdccptn36y0h40000gn/T/ipykernel_82345/1718940670.py:145: RuntimeWarning: invalid value encountered in divide
  cv2.imshow('Disparity', (disparity - disparity.min()) / (disparity.max() - disparity.min()))
2025-04-13 22:58:03.308 python[82345:28839113] +[IMKClient subclass]: chose IMKClient_Modern
2025-04-13 22:58:03.308 python[82345:28839113] +[IMKInputSession subclass]: chose IMKInputSession_Modern


: 